In [ ]:
import sqlite3


conn = sqlite3.connect(":memory:")

cursor = conn.cursor()


cursor.execute("""
    CREATE TABLE predictions (
        id INTEGER PRIMARY KEY,
        model_name TEXT,
        input_text TEXT,
        confidence REAL
    )
""")


cursor.execute("""
    INSERT INTO predictions (model_name, input_text, confidence)
    VALUES ('gpt-4', 'classify this email', 0.97)
""")

data = [
    ('yolo-v8', 'detect objects in image', 0.88),
    ('bert', 'classify this email', 0.76),
    ('gpt-4', 'summarize this document', 0.95),
]

cursor.executemany("""
    INSERT INTO predictions (model_name, input_text, confidence)
    VALUES (?, ?, ?)
""", data)

cursor.execute("""
    SELECT model_name, confidence
    FROM predictions
    WHERE confidence > 0.90
""")

cursor.execute("""
    SELECT model_name, AVG(confidence), COUNT(*)
    FROM predictions
    GROUP BY model_name
""")



cursor.execute("""
    SELECT model_name, AVG(confidence) AS avg_confidence, COUNT(*) AS total_runs
    FROM predictions
    GROUP BY model_name
    ORDER BY avg_confidence DESC
""")

cursor.execute("""
  select *
  from predictions
  WHERE model_name = 'gpt-4'
  ORDER BY confidence DESC
  LIMIT 1
""")

print(cursor.fetchall())

[(1, 'gpt-4', 'classify this email', 0.97)]


In [ ]:
pip install chromadb

In [ ]:
import chromadb

client = chromadb.Client()

collection = client.create_collection("ai_documents")

In [ ]:
collection.add(
    documents=[
        "how to fix a leaking pipe",
        "repairing a broken water tube",
        "machine learning model training guide",
        "neural networks and deep learning",
        "cooking pasta with tomato sauce"
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"]
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 30.6MiB/s]


In [ ]:
results = collection.query(
    query_texts=["broken pipe repair"],
    n_results=2
)
print(results['documents'])

[['repairing a broken water tube', 'how to fix a leaking pipe']]


In [ ]:
all_data = collection.get(include=["embeddings"])
print(len(all_data["embeddings"][0]))

384


**RAG**

In [ ]:
pip install langchain --upgrade chromadb langchain-community
pip install pypdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/CV of B.Sc. Eng. Nafiz Khan.pdf")

pages = loader.load()

In [ ]:
print(len(pages))

print(pages[0].page_content,pages[0].metadata)
print(pages[1].page_content)

2
Nafiz Khan Tasnul 
📧  www.nafizkhan.com@gmail.com |📞 01540134683| 📍 Khilgaon, Dhaka| 
🔗 https://github.com/Naf-Second | 
🔗 https://www.linkedin.com/in/nafiz-khan-a10726367/ | 
🔗 https://codeforces.com/profile/Secondo_is_nub 
 
  
Work Experience 
 
FPT Information System           Feb. 2025 – Present 
Junior Developer             Banani, Dhaka 
 
As a junior developer, I work on developing modules in an existing ERP (IVAS) system. 
 Developing module pool programs in an existing ERP system based on SAP TRM module 
 Hands-on experience with SAP ALV, ODATA, ABAP Custom Coding 
 Understood legacy code and implemented necessary modifications 
 Designed table controls and sub-screens with PBO/PAI logic 
 Dialog programming, validations, and modular screen handling 
 Debugged backend issues with the help of Breakpoints 
 Minor enhancements through Badi 
 
Research and Publications 
 
Custard Apple Disease Classification with Explainable AI 
 A Deep Learning based research project (

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 250, chunk_overlap = 50)

chunks = text_splitter.split_documents(pages)

print(len(chunks))
print(chunks[0])


13
page_content='Nafiz Khan Tasnul 
📧  www.nafizkhan.com@gmail.com |📞 01540134683| 📍 Khilgaon, Dhaka| 
🔗 https://github.com/Naf-Second | 
🔗 https://www.linkedin.com/in/nafiz-khan-a10726367/ | 
🔗 https://codeforces.com/profile/Secondo_is_nub 
 
  
Work Experience' metadata={'producer': 'iLovePDF', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-02-25T07:34:07+06:00', 'author': 'USER', 'moddate': '2026-02-25T01:36:01+00:00', 'source': '/content/CV of B.Sc. Eng. Nafiz Khan.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


In [ ]:
pip install langchain-openai
pip install sentence-transformers langchain-huggingface

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name ="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db"
)

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
result = retriever.invoke("give me his email")

for res in result:
  print(res.page_content)
  print("End of chunk\n")



Nafiz Khan Tasnul 
📧  www.nafizkhan.com@gmail.com |📞 01540134683| 📍 Khilgaon, Dhaka| 
🔗 https://github.com/Naf-Second | 
🔗 https://www.linkedin.com/in/nafiz-khan-a10726367/ | 
🔗 https://codeforces.com/profile/Secondo_is_nub 
 
  
Work Experience
End of chunk

Nafiz Khan Tasnul 
📧  www.nafizkhan.com@gmail.com |📞 01540134683| 📍 Khilgaon, Dhaka| 
🔗 https://github.com/Naf-Second | 
🔗 https://www.linkedin.com/in/nafiz-khan-a10726367/ | 
🔗 https://codeforces.com/profile/Secondo_is_nub 
 
  
Work Experience
End of chunk



In [ ]:
import os

os.environ["GROQ_API_KEY"] = "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"

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.3)

In [ ]:
pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.8 MB/s eta 0:00:00


In [ ]:
 from langchain_core.prompts import ChatPromptTemplate
 from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.

Context: {context}

Question: {question}
""")

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)

response = chain.invoke("I am thinking of testing his ability, like a take home project of 3 days")

print(response.content)

It seems like you're considering Nafiz Khan for a project or task. Based on his CV, it appears that he has experience as a junior developer, working on developing modules in an existing ERP (IVAS) system. 

Given his background, a 3-day take-home project might be suitable for assessing his skills. You could consider assigning a task that requires him to develop a module or a small application within a similar ERP system, or something related to his area of expertise. This would allow you to evaluate his problem-solving skills, ability to work independently, and technical proficiency.
